# 07 — Forecasting Baselines & Linear Benchmarks
**Persistence, Seasonal Naive, and Ridge Regularized Regression**

Establishes rigorous minimum performance baselines:
- **Persistence**: $\hat{y}_{t+h} = y_t$
- **Seasonal Naive**: $\hat{y}_{t+h} = y_{t+h-365}$
- **Ridge Regression**: Linear shrinkage with $L_2$ regularization.

In [ ]:
import sys, os, pandas as pd, numpy as np
sys.path.insert(0, os.path.abspath('..'))
from pipelines.preprocess_data import run_preprocessing_pipeline
from src.forecasting import rolling_origin_cross_validation

train_df, test_df = run_preprocessing_pipeline(sample_mode=False)
full_df = pd.concat([train_df, test_df], ignore_index=True)

feature_cols = [
    'month_sin', 'month_cos',
    'depth_to_water_mbgl_lag_1',
    'depth_to_water_mbgl_roll_mean_3',
    'specific_yield_est',
    'recharge_potential_index',
    'stage_of_extraction_pct'
]

mean_depth = float(full_df['depth_to_water_mbgl'].dropna().mean() or 12.0)
for col in ['depth_to_water_mbgl_lag_1', 'depth_to_water_mbgl_roll_mean_3']:
    full_df[col] = full_df[col].fillna(mean_depth)
for col in ['specific_yield_est', 'recharge_potential_index', 'stage_of_extraction_pct']:
    full_df[col] = full_df[col].fillna(full_df[col].median())

results = rolling_origin_cross_validation(full_df, feature_cols, n_splits=3)
display(pd.DataFrame(results).T)